## Task 3.1: Run the Agents Notebook

### Scenario Context

This lab simulates a conversation that is already in progress. The agent has previously interacted with the user to gather preferences and constraints. You're now observing the agent as it continues assisting with product recommendations.

This mirrors a multi-turn workflow where agents rely on memory and context from earlier interactions. It sets the stage for deeper analysis of planning behavior and reasoning strategy.

In this notebook, you use a shopping assistant agent to answer questions about lawn maintenance products from two companies - AnyCompany Outdoor Power Equipment and AnyCompany LawnCare Solutions. Execute the cells in this notebook to simulate a chat conversation with the agent, to test the prebuilt knowledge base, guardrails, and the agent application. 

The knowledge base includes product details such as manufacturer, description, and rating. The agent can access functions that support simple calculations and price lookup. 

In the following steps, you list the agents and invoke the shopping assistant agent with session and memory parameters to understand the impact of short-term and the longer-term memory on the agent's behavior. You also capture and study the trace to understand the functioning of the agent including the use of a shopping assistant with guardrails.

Advance through each cell of the notebook, running each code cell and viewing its output.

### Task 3.1.1: Install Boto3 and import required Python libraries

In this task, you import Boto3 as well as supporting Python libraries. You also import the Boto3 Client error-handling library, configure the Session, AWS Region and Bedrock clients and helper Agent functions.

The next cell imports the Python Boto3 library. 
The AWS SDK for Python (Boto3) provides a Python API for AWS infrastructure services. Using the SDK for Python, you can build applications on top of AWS Services. Boto3 library to interact with Amazon Bedrock Agents and Foundation Models.

In [1]:
import boto3
import json
import gzip
import shutil
import time
import logging
import pprint
import json
import uuid
from datetime import datetime

Import the Boto3 Client error-handling library.

In [2]:
from botocore.exceptions import ClientError

### Task 3.1.2: Configure Boto3 objects and helper functions 

Configure the Session, AWS Region and Bedrock clients.

In [3]:
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime')
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

Setup a helper function to find agent id by agent name.

In [4]:
def find_agent_id_by_agent_name(client, agent_name):
    try:
        response = client.list_agents()
        agents = response.get('agentSummaries', [])
    except ClientError as e:
        print(f"Error listing agents: {e}")
        return []
    agent_id = ''
    for agent in agents:
        if agent['agentName'] == agent_name:
            agent_id = agent['agentId']
    return agent_id

Invoke the helper function find_agent_id_by_agent_name to get the agent id associated with the shopping-assistant agent. In the below cell, you will see that an **alias_id** is defined. This **alias_id** will be used in later steps when the Session Summary is requested. 

In [5]:
# Invoke the helper function to find agent IDs
agent_id = find_agent_id_by_agent_name(bedrock_agent_client, 'shopping-assistant')
agent_id_with_guardrail = find_agent_id_by_agent_name(bedrock_agent_client, 'shopping-assistant-with-guardrails')

# Get the agent alias ID
try:
    # Try to get the alias ID for the shopping-assistant agent
    response = bedrock_agent_client.list_agent_aliases(agentId=agent_id)
    if response.get('agentAliasSummaries'):
        alias_id = response['agentAliasSummaries'][0]['agentAliasId']
    else:
        # If no alias found for shopping-assistant, try with the guardrail agent
        response = bedrock_agent_client.list_agent_aliases(agentId=agent_id_with_guardrail)
        if response.get('agentAliasSummaries'):
            alias_id = response['agentAliasSummaries'][0]['agentAliasId']
        else:
            # Default fallback if no aliases found
            alias_id = "DRAFT"
except Exception as e:
    print(f"Error getting agent alias: {e}")
    # Use DRAFT as fallback - this is a common default alias name in Bedrock
    alias_id = "DRAFT"

print(f"Using agent alias ID: {alias_id}")

# the below s3 bucket will be used internally to store logs for bedrock
bedrock_logging_bucket="bedrock-logging-"+region+"-"+account_id
recent_bedrock_log_file_copy='recent_bedrock_log_file_copy'

Using agent alias ID: TSTALIASID


Create a helper function to invoke the Bedrock Agent and capture memory.

In [6]:
def invokeAgent(query, session_id, memory_id, agent_id, enable_trace=False, session_state=dict(), end_session=False):
    
    if memory_id == "":
        agentResponse = bedrock_agent_runtime_client.invoke_agent(
            inputText=query,
            agentId=agent_id,
            agentAliasId=alias_id, 
            sessionId=session_id,
            enableTrace=enable_trace, 
            endSession= end_session,
            sessionState=session_state
        )
    else:
    # invoke the agent API
        agentResponse = bedrock_agent_runtime_client.invoke_agent(
            inputText=query,
            agentId=agent_id,
            agentAliasId=alias_id, 
            memoryId=memory_id,
            sessionId=session_id,
            enableTrace=enable_trace, 
            endSession= end_session,
            sessionState=session_state
        )
    
    if enable_trace:
        # Create a safe copy of the response for logging, handling datetime objects
        try:
            # Log only the metadata, not the event stream
            safe_response = {k: str(v) for k, v in agentResponse.items() if k != 'completion'}
            logger.info(f"Agent response metadata: {safe_response}")
        except Exception as e:
            logger.info(f"Could not print full response due to: {str(e)}")
    
    event_stream = agentResponse['completion']
    try:
        for event in event_stream:        
            if 'chunk' in event:
                data = event['chunk']['bytes']
                if enable_trace:
                    logger.info(f"Final answer ->\n{data.decode('utf8')}")
                agent_answer = data.decode('utf8')
                end_event_received = True
                return agent_answer
                # End event indicates that the request finished successfully
            elif 'trace' in event:
                if enable_trace:
                    # Handle potential datetime objects in trace
                    try:
                        trace_str = json.dumps(event['trace'], default=lambda o: str(o), indent=2)
                        logger.info(trace_str)
                    except Exception as e:
                        logger.info(f"Could not print trace due to: {str(e)}")
                        logger.info(str(event['trace']))
            else:
                raise Exception("unexpected event.", event)
    except Exception as e:
        if str(e).find("throttlingException"): 
            print("\033[1mThrottlingException:\033[;7m\nA throttling error occurred when calling the InvokeAgent operation. Please wait up to 60 seconds and retry this cell.\n\033[0m")

### Task 3.1.3: Interact with the Bedrock agent without guardrails  

When interacting and chatting with Bedrock agents, there are two components that help maintain the state:

**session_id** represents a conversation with an agent across multiple questions. To continue the same conversation with an agent, use the same session_id value in the request.  

**memory_id** is where each user's conversation history and context are securely stored, ensuring complete separation between users.  

Setup the **session_id** and **memory_id**. 

In [7]:
session_id:str = str(uuid.uuid4())
print("Session id="+session_id)
memory_id_1:str = str(uuid.uuid4())
print("Memory id="+memory_id_1)

Session id=53ecd93c-0586-4aa8-aa97-f90e16f74863
Memory id=7902c0da-0d30-4d91-9fe0-516331aa0306


Simulate a dialog with the agent. Ask an initial question to the agent to get the conversation started. Some of the questions to the agent will be answered from information stored in the Knowledge Base. Since logging is active, you will notice latency. Notice that the **shopping-assistant** agent is being invoked. This agent does not have any associated Guardrails.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** This lab uses a non-production AWS account with a model invocation rate limit. If any of the subsequent cells that post questions to the agent returns a **Throttling Error**, wait up to 60 seconds and rerun that cell. 

In [8]:
q="Hello, I am interested in your products. Who manufactures the products you sell?"
print("\033[1mQuestion:\033[0m "+q+"\n")
try: 
    response = invokeAgent(q, session_id,memory_id_1, agent_id)
    print("\033[1mResponse:\033[0m "+response+"\n")
except Exception as e:
    print(f"Error: {str(e)}")

Question: Hello, I am interested in your products. Who manufactures the products you sell?

Response: Based on the search results, we have products from two main manufacturers:

1. AnyCompany Outdoor Power Equipment - They manufacture:
   - String Trimmer
   - Hedge Trimmer
   - Leaf Blower

2. AnyCompany LawnCare Solutions - They manufacture:
   - Lawn Mower
   - Aerator

These manufacturers specialize in outdoor and lawn maintenance equipment, offering battery-operated and gas-powered tools for various gardening and landscaping needs.



Continue asking questions about products. Execute each question and review the response. 

In [9]:
q="What do customers say about these products?"
print("\033[1mQuestion:\033[0m "+q+"\n")
try: 
    response = invokeAgent(q, session_id,memory_id_1, agent_id)
    print("\033[1mResponse:\033[0m "+response+"\n")
except Exception as e:
    print(f"Error: {str(e)}")

Question: What do customers say about these products?

Response: Here's a breakdown of customer ratings for the products:

AnyCompany Outdoor Power Equipment:
1. String Trimmer: 4 stars out of 5 (126 reviews)
2. Hedge Trimmer: 3.5 stars out of 5 (78 reviews)
3. Leaf Blower: 4.5 stars out of 5 (34 reviews)

AnyCompany LawnCare Solutions:
1. Lawn Mower: 4.5 stars out of 5 (20 reviews)
2. Aerator: 3.5 stars out of 5 (12 reviews)

Overall, customers seem to be generally satisfied with these products, with most items receiving ratings between 3.5 and 4.5 stars. The Lawn Mower and Leaf Blower appear to be the most highly rated products, both scoring 4.5 out of 5 stars.



In [10]:
q="How much do they cost?"
print("\033[1mQuestion:\033[0m "+q+"\n")
try: 
    response = invokeAgent(q, session_id,memory_id_1, agent_id)
    print("\033[1mResponse:\033[0m "+response+"\n")
except Exception as e:
    print(f"Error: {str(e)}")

Question: How much do they cost?

Response: Here are the prices for the products:

AnyCompany Outdoor Power Equipment:
1. String Trimmer (P001): $50
2. Hedge Trimmer (P002): $60
3. Leaf Blower (P003): $70

AnyCompany LawnCare Solutions:
1. Lawn Mower (Y001): $1,200
2. Aerator (Y002): $800

The outdoor power tools are relatively affordable, ranging from $50 to $70, while the larger lawn care equipment like the riding lawn mower and aerator are more expensive, priced at $800 and $1,200 respectively.



Ask another question, that will prompt the agent to use a math calculation.

In [11]:
q="How much will it cost to buy 2 string trimmers?"
print("\033[1mQuestion:\033[0m "+q+"\n")
try: 
    response = invokeAgent(q, session_id,memory_id_1, agent_id)
    print("\033[1mResponse:\033[0m "+response+"\n")
except Exception as e:
    print(f"Error: {str(e)}")

Question: How much will it cost to buy 2 string trimmers?

Response: The cost of 2 string trimmers would be $100. Each string trimmer is priced at $50, so 2 of them would total $100.



Ask a final question in this session. Since you are calling the **shopping-assistant** agent without Guardrails, ask a question that will not be blocked.

In [12]:
q="Can I use a string trimmer to control weeds?"
print("\033[1mQuestion:\033[0m "+q+"\n")
try: 
    response = invokeAgent(q, session_id,memory_id_1, agent_id)
    print("\033[1mResponse:\033[0m "+response+"\n")
except Exception as e:
    print(f"Error: {str(e)}")

Question: Can I use a string trimmer to control weeds?

Response: 
While the product description mentions the string trimmer is useful for "curb appeal", it doesn't specifically detail its use for weed control. String trimmers are typically used for trimming grass and weeds around edges, fences, and hard-to-reach areas where lawn mowers can't easily go. They can help manage weeds by cutting them down, but they don't remove the roots.




For comprehensive weed control, you might want to combine the string trimmer with other methods like manual weeding or using herbicides. The string trimmer is best for maintaining edges and keeping weeds from growing too tall, but it won't completely eliminate them.




The code below closes the session by setting **end_session=True** so that you can access the session summary later from the memory store.  

In [13]:
try:
    response = invokeAgent("end session", session_id,memory_id_1, agent_id, end_session=True)
    print("\033[1mResponse:\033[0m "+response+"\n")
except Exception as e:
    print(f"Error: {str(e)}")

Response: Session is terminated as 'endSession' flag is set in request.



### Task 3.1.4: Interact with the Bedrock agent's invocation logs


Define a set of helper functions to locate, download and save the recent Bedrock invocation log and print the json content in the downloaded gzipped file. This approach reduces the amout of output on the screen. The other option to get the invocation log is to enable trace while asking the agent a question. You will enable trace when using the agent with guardrail.  

In [14]:
def find_the_recent_bedrock_log():
    # List the objects in the bucket
    response = s3_client.list_objects_v2(Bucket=bedrock_logging_bucket)

    # Sort the objects by their LastModified attribute in descending order
    sorted_objects = sorted(response['Contents'], key=lambda obj: obj['LastModified'], reverse=True)

    # The first object in the sorted list is the most recent one
    latest_object = sorted_objects[0]

    # return the key (name) of the latest object
    print(latest_object['Key'])
    return latest_object['Key']

In [15]:
def download_recent_bedrock_log():
    # wait one minute for bedrock log to show up
    time.sleep(60)

    #set up file names
    recent_log_file_gz=recent_bedrock_log_file_copy+".gz"
    recent_log_file_json=recent_bedrock_log_file_copy+".json"

    # download the log to local directory
    s3_client.download_file(bedrock_logging_bucket, find_the_recent_bedrock_log(), recent_log_file_gz)

    #unzip the .gz file
    with gzip.open(recent_log_file_gz, 'rb') as f_in:
        with open(recent_log_file_json, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    return recent_log_file_json

In [16]:
def download_and_print_json_log(output_file:str):
    try:
        log_name=download_recent_bedrock_log()
        with open(log_name, 'r') as file:
            lines = file.readlines()
            with open(output_file, 'w') as f_out:
                for line in lines:
                    # Handle potential datetime objects in JSON
                    try:
                        parsed_json = json.loads(line.strip())
                        json.dump(parsed_json, f_out, indent=2, default=lambda o: str(o))
                    except Exception as e:
                        print(f"Error processing log line: {str(e)}")
                        f_out.write(line)  # Write the original line if parsing fails
        print(f"Log successfully written to {output_file}")
    except Exception as e:
        print(f"Error downloading or processing log: {str(e)}")

Print the trace for the recent agent invocation. The code sleeps for one minute to ensure that the latest logs are fetched. 

In [17]:
download_and_print_json_log("log_output_without_guardrails.txt")

logs/AWSLogs/449362132532/BedrockModelInvocationLogs/us-west-2/2025/10/02/11/20251002T113717836Z_73c8768b8b68ba1a.json.gz
Log successfully written to log_output_without_guardrails.txt


The log output shows the reasoning used by the agent to answer user's questions. Take a few minutes to study the log.

### Task 3.1.5: Interact with the Bedrock agent with guardrails  

Pose the last question seeking weed advice to the **shopping-assistant-with-guardrails** agent. Notice the agent refuses to provide lawn maintenance advice for the last question in the list above. Since enable_trace is set to true, you can see the trace output on the screen. The trace output is available in the log file as well. 

In [18]:
session_id:str = str(uuid.uuid4())
memory_id=""
q="Can I use a string trimmer to control weeds?"

print("\033[1mQuestion:\033[0m "+q+"\n")
try: 
    response = invokeAgent(q, session_id,memory_id, agent_id_with_guardrail, enable_trace=True)
    print("\033[1mResponse:\033[0m "+response+"\n")
except Exception as e:
    print(f"Error: {str(e)}")

Question: Can I use a string trimmer to control weeds?



[2025-10-02 11:38:04,310] p1498 {1319702912.py:31} INFO - Agent response metadata: {'ResponseMetadata': "{'RequestId': '0a5040c1-3bf3-419c-8793-87003c08b9c8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 02 Oct 2025 11:38:04 GMT', 'content-type': 'application/vnd.amazon.eventstream', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'x-amzn-requestid': '0a5040c1-3bf3-419c-8793-87003c08b9c8', 'x-amz-bedrock-agent-session-id': 'ecc1e70e-ee77-4658-a482-588cf202ad35', 'x-amzn-bedrock-agent-content-type': 'application/json'}, 'RetryAttempts': 0}", 'contentType': 'application/json', 'sessionId': 'ecc1e70e-ee77-4658-a482-588cf202ad35'}
[2025-10-02 11:38:04,819] p1498 {1319702912.py:51} INFO - {
  "sessionId": "ecc1e70e-ee77-4658-a482-588cf202ad35",
  "trace": {
    "guardrailTrace": {
      "action": "INTERVENED",
      "traceId": "0a5040c1-3bf3-419c-8793-87003c08b9c8-guardrail-pre-0",
      "inputAssessments": [
        {
          "topicPolicy": {
            "topics": [
 

Response: Sorry, your query violates our usage policies. We do not provide lawn maintenance advice. To discuss the best options to maintain your lawn, please contact a lawn professional.



Print the log for the recent agent invocation. The code sleeps for one minute to ensure that the latest logs are fetched. 

In [24]:
download_and_print_json_log("log_output_with_guardrails.txt")

logs/AWSLogs/449362132532/BedrockModelInvocationLogs/us-west-2/2025/10/02/11/20251002T114548047Z_d82ae3229dd0d343.json.gz
Log successfully written to log_output_with_guardrails.txt


### Reasoning and Planning Strategy

Let’s analyze how the agent reasoned through the task.


**Action Chain Summary:**

- Interpreted user intent from input
- Matched intent to appropriate tool or action
- Queried external function or retrieved memory
- Constructed a final response

**Observations:**

- Memory enabled the agent to reference earlier parts of the conversation
- Guardrails prevented sensitive or inappropriate responses
- The agent exhibited planner-like behavior, chaining steps together implicitly

### Reflection: Memory and Guardrails


Reflect on the agent’s behavior using the questions below:


- How did memory influence the agent’s response?
- Were any responses modified or blocked due to guardrails?
- How would the agent behave differently if memory were disabled?


#### Check all that apply:

- [ ] I understand how the agent retained context across steps.
- [ ] I noticed where guardrails prevented unsafe output.
- [ ] I can see how memory supports multi-turn workflows

### Task 3.1.6: Interact with the agent's session memory

More than two minutes have elapsed since closing the first session. The memory store with a summary of the first session should be ready by now. This can be useful for auditing, debugging, or understanding the agent's reasoning across multiple conversations. The below function checks if the Session Summary is finalized, printing the Session Summary. If the Summary is not yet available, the code below will wait 30 seconds and retry up to 5 times. After running this cell, if the Summary is still not available, you can wait one more minute and rerun this cell. 

In [22]:
import time

def check_session_summary(bedrock_agent_runtime_client, agent_id, alias_id, memory_id, max_retries=5, wait_time=30):
    for attempt in range(max_retries):
        try:
            response = bedrock_agent_runtime_client.get_agent_memory(
                agentId=agent_id,
                agentAliasId=alias_id,
                memoryId=memory_id,
                memoryType='SESSION_SUMMARY'
            )
            
            if 'memoryContents' in response and response['memoryContents']:
                print("Summary found!")
                return response['memoryContents'][0]['sessionSummary']['summaryText']
            
            print(f"Attempt {attempt + 1}/{max_retries}: Summary not yet generated. Waiting {wait_time} seconds...")
            time.sleep(wait_time)
            
        except Exception as e:
            print(f"Error checking summary: {str(e)}")
            return None
    
    print("Summary not found after all retries. The summary might take longer to generate.")
    print("Please wait 60 seconds and rerun this cell.")
    return None

# Usage example
try:
    summary = check_session_summary(
        bedrock_agent_runtime_client,
        agent_id=agent_id,
        alias_id=alias_id,
        memory_id=memory_id_1
    )

    if summary:
        print("\nSession Summary:")
        print(summary)
except Exception as e:
    print(f"Error retrieving session summary: {str(e)}")

Summary found!

Session Summary:
 - Inquire about product manufacturers
- Learn about customer reviews and ratings for outdoor power equipment
- Find out pricing for string trimmers and other products
- Understand if a string trimmer can be used for weed control - Used x_amz_knowledgebase_WG3VBMGENJ tool to retrieve information about:
  * Manufacturers of products
  * Customer reviews and ratings
- Used PriceLookup tool to find prices for:
  * String Trimmer (P001): $50
  * Hedge Trimmer (P002): $60
  * Leaf Blower (P003): $70
  * Lawn Mower (Y001): $1,200
  * Aerator (Y002): $800
- Used MultiFunctionCalculatorTool to calculate cost of 2 string trimmers: $100
- Provided information about string trimmers and weed control - Manufacturers: AnyCompany Outdoor Power Equipment and AnyCompany LawnCare Solutions
- Products include: String Trimmer, Hedge Trimmer, Leaf Blower, Lawn Mower, Aerator
- Most products are battery-operated or gas-powered
- Product ratings range from 3.5 to 4.5 stars


Finish this conversation with the agent by asking it to recall what was discussed. 

In [23]:
q='What was I interested in buying?'
print("\033[1mQuestion:\033[0m "+q+"\n")
try: 
    response = invokeAgent(q, session_id,memory_id_1, agent_id)
    print("\033[1mResponse:\033[0m "+response+"\n")
except Exception as e:
    print(f"Error: {str(e)}")

Question: What was I interested in buying?

Response: Based on our previous conversation, you were interested in purchasing outdoor power equipment, specifically:
- String Trimmers
- Hedge Trimmers
- Leaf Blowers
- Lawn Mowers
- Aerators

These products were from manufacturers like AnyCompany Outdoor Power Equipment and AnyCompany LawnCare Solutions.



### Bedrock Best Practices for Agents

Here are some recommended best practices based on how the agent was implemented in this lab:

**Memory**

- Limit memory retention to what's necessary per user/session
- Avoid storing sensitive or personal data
- Review and clear memory between sessions if needed

<i aria-hidden="true" class="fas fa-info-circle" style="color:#007FAA"></i> **Learn more:** [Retain conversational context across multiple sessions using memory](https://docs.aws.amazon.com/bedrock/latest/userguide/agents-memory.html)

**Guardrails**

- Configure content filters and denied topics
- Regularly review system responses to ensure safe output

<i aria-hidden="true" class="fas fa-info-circle" style="color:#007FAA"></i> **Learn more:** [Detect and filter harmful content by using Amazon Bedrock Guardrails](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html)

**Observability**

- Enable agent logs to analyze behavior and trace decisions
- Use logging tools for debugging and performance evaluation

<i aria-hidden="true" class="fas fa-info-circle" style="color:#007FAA"></i> **Learn more:** [Monitoring the performance of Amazon Bedrock
](https://docs.aws.amazon.com/bedrock/latest/userguide/monitoring.html)

### Cleanup

You have completed this notebook. To move to the next part of the lab, do the following:
- Close this notebook file.
- Return to the lab session and continue with the **Conclusion**.